# 43-1 Preprocesamiento de Texto 2 — Stemming, Lematización y N-gramas

Este notebook demuestra técnicas avanzadas de preprocesamiento de texto usando NLTK (via UDFs) y PySpark ML, sin dependencias de Spark NLP (John Snow Labs).

## Objetivo
Aplicar stemming, lematización, eliminación de stopwords y generación de n-gramas sobre un conjunto de palabras y oraciones de ejemplo.

## Flujo del notebook
1. Instalación de dependencias (NLTK, visualización).
2. Importación de módulos PySpark y NLTK.
3. Creación de la sesión Spark.
4. Creación de datos de ejemplo (palabras con variaciones morfológicas).
5. Stemming con NLTK `SnowballStemmer` via UDF.
6. Lematización con NLTK `WordNetLemmatizer` via UDF.
7. Eliminación de stopwords y generación de bigramas con PySpark ML.

## Antes de ejecutar
- Verifica que la sesión de Spark esté activa.
- Ejecuta las celdas en orden para mantener dependencias.


## 1. Instalación de dependencias
Se instalan librerías para visualización, manejo de datos y procesamiento de texto con NLTK. No se requiere Spark NLP.


In [ ]:
%pip install numpy pandas matplotlib seaborn wordcloud pypdf2 contractions unidecode nltk

## 2. Importación de módulos
Se importan los módulos de PySpark (sesión, tipos, funciones) y `Pipeline` de PySpark ML.


In [ ]:
import findspark
findspark.init('/Users/joseaguilar/Documents/Development/spark/spark-3.5.1-bin-hadoop3')
from pyspark import *
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField, StringType, IntegerType, DateType, TimestampType, LongType
from pyspark.sql.types import ArrayType, DoubleType, BooleanType, DecimalType
from pyspark.sql.functions import regexp_extract, split, from_unixtime, col, avg, min, max
from pyspark.sql.functions import grouping_id, window, explode, to_json, from_json
from pyspark.sql.functions import udf, lit, current_timestamp, current_date, date_format
from pyspark.ml import PipelineModel, Pipeline
import PyPDF2
import os

## 3. Creación de la sesión Spark
Se inicializa una `SparkSession` básica sin dependencias externas de NLP.


In [ ]:
spark = SparkSession.builder \
    .appName("preprocesamiento_texto_2") \
    .getOrCreate()

## 4. Descarga de recursos NLTK e importaciones
Se descargan los datos necesarios de NLTK (`punkt`, `wordnet`, `omw-1.4`) y se importan `SnowballStemmer` y `WordNetLemmatizer` para usarlos como UDFs.


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.stem import SnowballStemmer
from nltk.stem import WordNetLemmatizer
from pyspark.sql.functions import lower, regexp_replace, trim, split, explode, col, udf
from pyspark.sql.types import StringType, ArrayType

print(f"Spark version: {spark.version}")
import sys
print(f"Python version: {sys.version}")

## 5. Datos de ejemplo
Se crea un DataFrame con palabras en inglés que tienen variaciones morfológicas (conjugaciones, plurales, comparativos) para demostrar stemming y lematización.


In [ ]:
palabras_ejemplo = [
    ("computation",), ("computer",), ("computing",),
    ("running",), ("ran",), ("run",),
    ("playing",), ("played",), ("play",),
    ("better",), ("best",), ("good",),
    ("studies",), ("study",), ("studying",),
    ("wolves",), ("wolf",), ("wolfing",),
]
df_palabras = spark.createDataFrame(palabras_ejemplo, ["palabra"])
df_palabras.show()

## 6. Stemming con NLTK SnowballStemmer
Se aplica stemming (reducción a la raíz morfológica) usando `SnowballStemmer` de NLTK a través de una UDF de PySpark. Esto reduce palabras como "computing" → "comput", "running" → "run".


In [ ]:
# Definir UDF de stemming usando NLTK SnowballStemmer
stemmer_en = SnowballStemmer("english")

@udf(StringType())
def stem_word(word):
    if word is None:
        return None
    return stemmer_en.stem(word)

# Aplicar stemming a cada palabra
df_stemmed = df_palabras.withColumn("palabra_stem", stem_word(col("palabra")))
df_stemmed.show(truncate=False)

## 7. Lematización con NLTK WordNetLemmatizer
Se aplica lematización (reducción a la forma base del diccionario) usando `WordNetLemmatizer` de NLTK via UDF. A diferencia del stemming, la lematización produce palabras reales: "wolves" → "wolf", "better" → "better", "studies" → "study".

Se muestra una comparación entre la palabra original, su stem y su lema.


In [ ]:
# Definir UDF de lematización usando NLTK WordNetLemmatizer
lemmatizer_en = WordNetLemmatizer()

@udf(StringType())
def lemmatize_word(word):
    if word is None:
        return None
    # Intentar lematizar como verbo, sustantivo y adjetivo; devolver el más corto
    lemmas = [
        lemmatizer_en.lemmatize(word, pos='v'),  # verbo
        lemmatizer_en.lemmatize(word, pos='n'),  # sustantivo
        lemmatizer_en.lemmatize(word, pos='a'),  # adjetivo
    ]
    return min(lemmas, key=len)

# Aplicar lematización y comparar con stemming
df_lemmas = df_stemmed.withColumn("lematizado", lemmatize_word(col("palabra")))
df_lemmas.select("palabra", "palabra_stem", "lematizado").show(truncate=False)

## 8. Eliminación de stopwords y generación de bigramas
Se crean oraciones de ejemplo, se tokenizan con `split`, se eliminan stopwords con `StopWordsRemover` de PySpark ML y se generan bigramas con `NGram`.


In [ ]:
from pyspark.ml.feature import NGram, StopWordsRemover

# Crear algunos textos de ejemplo para n-gramas
oraciones = [("to be or not to be",), ("all your base are belong to us",)]
df_oraciones = spark.createDataFrame(oraciones, ["texto"])

# 1. Tokenizar con split nativo de PySpark
df_tokenized = df_oraciones.withColumn("tokens", split(lower(col("texto")), r"\s+"))

# 2. Eliminar stopwords con StopWordsRemover de PySpark ML
custom_stopwords = ["y", "de", "la", "the", "and", "is", "es", "porque",
                    "to", "or", "not", "are", "a", "an"]
stopwords_remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="tokens_final",
    stopWords=custom_stopwords,
    caseSensitive=False
)
df_tokens = stopwords_remover.transform(df_tokenized)

# 3. Generar bigramas
ngrammer = NGram(n=2, inputCol="tokens_final", outputCol="bigrams")
df_bigrams = ngrammer.transform(df_tokens)
df_bigrams.select("tokens_final", "bigrams").show(truncate=False)